# 👗 DressApp Eyes (Gemma-4 E4B) Multilingual Multimodal Fine-Tuning & Quantization Pipeline

This notebook provides the complete end-to-end pipeline to fine-tune **DressApp Eyes** (`google/gemma-4-e4b-it`) across all fashion workflows in all **13 supported languages** (`en`, `he`, `ar`, `es`, `fr`, `de`, `it`, `pt`, `ru`, `zh`, `ja`, `hi`, `nl`) using a **Tri-Dataset Hybrid Blend** from Hugging Face:
1. **`ashraq/fashion-product-images-small`**: Clean isolated studio product cutouts for exact taxonomy grounding (Tops, Bottoms, Footwear, Accessories, Outerwear/Dresses).
2. **`Marqo/deepfashion-multimodal`**: High-resolution garments with detailed descriptions for complex patterns (camo, plaid, floral), rich fabric materials (silk, linen, denim, wool), and fluent captions.
3. **`ArtmeScienceLab/Garments2Look`**: 80,000 multi-garment to complete look pairs for full-body outfit coordination (mandatory top+bottom OR dress + footwear) and anatomical ordering.

### Key Improvements in this Release:
- **13-Language Multilingual Grounding**: Generates authentic localized fashion terminology across all 13 supported languages while preserving strict English schema keys.
- **Native Gemma-4 Chat Template**: Formats inputs with `<start_of_turn>user` / `<start_of_turn>model` syntax instead of legacy raw strings.
- **Prompt Loss Masking**: Masks prompt/image tokens (`labels[:prompt_len] = -100`) so gradient updates only optimize the assistant JSON response, eliminating formatting collapse.
- **Pinned Dependencies**: Resolves protobuf and HuggingFace Hub version collisions for Python 3.13 and bleeding-edge Transformers.

### Workflow Steps:
1. **Environment Setup & llama.cpp Build**: Install PyTorch, Transformers, PEFT, BitsAndBytes, and compile `llama.cpp` tools (`convert_hf_to_gguf.py` & `llama-quantize`).
2. **Import Assets from VPS (Optional)**: Connect to the Hetzner VPS (`178.105.144.142`) via SCP if needed.
3. **Tri-Dataset Ingestion & 13-Language Multilingual ETL**: Stream and format all three datasets directly into DressApp's canonical JSON schemas with exact user prompt directives.
4. **QLoRA Fine-Tuning with Loss Masking**: 4-bit quantized SFT training targeting attention and MLP projections with prompt masking.
5. **Quantize to GGUF**: Merge PEFT LoRA into base weights, export intermediate F16 GGUF, and quantize to `Q3_K_M` (~2.85 GB) and `Q4_K_M`.
6. **Evaluation & Smoke Tests**: Test attribute extraction, unique localized titles, 100% color/material sums, and outfit coordination.
7. **Push to VPS & Verify**: Securely upload `gemma-4-E4B-it-Q3_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` and restart `dressapp-eyes`.


## 🛠️ Step 1: Environment Setup & Tooling
Install deep learning dependencies, Hugging Face streaming libraries, and build `llama.cpp` conversion and quantization tools.


In [ ]:
# Check GPU availability
!nvidia-smi

# Remove incompatible legacy torchao and unused tensorflow to eliminate protobuf collisions
!pip uninstall -y torchao tensorflow

# Force disable TensorFlow in Transformers environment
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

# Install PyTorch with CUDA 12.1 and modern Python 3.13 scientific libraries
!pip install -q -U --force-reinstall 'pillow>=11.0.0'
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U 'numpy>=2.2.1' 'scipy>=1.15.0' 'pandas>=2.2.3' pyarrow
# Install deep learning and conversion dependencies with exact compatibility pins
!pip install -q -U 'huggingface_hub==1.33.0' 'tokenizers==0.23.2'
!pip install -q -U datasets peft 'bitsandbytes>=0.46.1' accelerate paramiko scp gguf sentencepiece protobuf
# Install transformers from git source LAST to ensure bleeding-edge Gemma-4 architecture
!pip install -q -U --force-reinstall git+https://github.com/huggingface/transformers.git

# Clone and build llama.cpp for GGUF conversion & quantization utilities
import os, sys
if not os.path.exists('llama.cpp'):
    !git clone https://github.com/ggerganov/llama.cpp.git
    !cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF
    !cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize

print('✅ Dependencies installed and llama.cpp tools compiled!')

# Automatic runtime restart if NumPy, Pillow, or Transformers was upgraded in an active session
import os, sys
needs_restart = False

# 1. Check Pillow typing compatibility before any torchvision/transformers imports
try:
    import PIL._typing
    if not hasattr(PIL._typing, '_Ink'):
        needs_restart = True
except Exception:
    needs_restart = True

if needs_restart:
    print('🔄 Triggering automatic kernel reload to activate fresh Pillow typing definitions...')
    os.kill(os.getpid(), 9)

# 2. Check NumPy 2.x ABI
try:
    import numpy as np
    np_ver = tuple(map(int, np.__version__.split('.')[:2]))
    if np_ver < (2, 2):
        needs_restart = True
except Exception:
    needs_restart = True

if needs_restart:
    print('🔄 Triggering automatic kernel reload to activate fresh NumPy 2.x...')
    os.kill(os.getpid(), 9)

# 3. Check bleeding-edge Transformers Gemma-4 support
try:
    import transformers
    if not hasattr(transformers, 'Gemma4Processor') and not hasattr(transformers, 'Gemma4ForConditionalGeneration'):
        needs_restart = True
except Exception:
    needs_restart = True

if needs_restart:
    print('🔄 Triggering automatic kernel reload to activate fresh Transformers with Gemma-4...')
    os.kill(os.getpid(), 9)

import numpy as np
import transformers
print(f'✅ NumPy {np.__version__}, Pillow, and Transformers {transformers.__version__} ready!')


## 📥 Step 2: (Optional) Import Assets from Hetzner VPS Server
> **Note:** This step is **OPTIONAL** and can be safely skipped! Proceed directly to Step 3.
> 
> You do **NOT** need `mmproj-BF16.gguf` during training or GGUF quantization. `mmproj-BF16.gguf` already exists on the VPS at `/srv/AI-Stylist/eyes_gguf/mmproj-BF16.gguf` and remains unchanged.


In [ ]:
import os
from pathlib import Path

LOCAL_ASSETS_DIR = Path('vps_assets')
LOCAL_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

print('ℹ️ Step 2 is OPTIONAL. You can safely proceed to Step 3 without downloading from VPS.')
print('   The mmproj projector is already live on the VPS and is not modified during QLoRA.')


## 📦 Step 3: Tri-Dataset Ingestion & 13-Language Multilingual ETL
Integrate all three Hugging Face datasets into DressApp's canonical schemas across all 13 supported languages (`en`, `he`, `ar`, `es`, `fr`, `de`, `it`, `pt`, `ru`, `zh`, `ja`, `hi`, `nl`):
1. **`ashraq/fashion-product-images-small`**: Studio cutouts for isolated item recognition.
2. **`Marqo/deepfashion-multimodal`**: Fine-grained textures, fabrics (silk/linen/wool/denim), and complex patterns (camo/plaid/houndstooth).
3. **`ArtmeScienceLab/Garments2Look`**: Multi-piece outfits for Scheduled Outfit head-to-toe coordination.


In [ ]:
# Defensive safeguard for datasets streaming on Python 3.13
try:
    import scipy.io
except Exception as _scipy_err:
    import sys, types
    print(f'ℹ️ Shielding datasets streaming patcher against scipy ABI mismatch: {_scipy_err}')
    _dummy_scipy = types.ModuleType('scipy')
    _dummy_scipy.__version__ = '1.15.0'
    _dummy_io = types.ModuleType('scipy.io')
    _dummy_io.__version__ = '1.15.0'
    _dummy_io.loadmat = lambda *args, **kwargs: {}
    _dummy_scipy.io = _dummy_io
    sys.modules['scipy'] = _dummy_scipy
    sys.modules['scipy.io'] = _dummy_io

from datasets import load_dataset
import json
from PIL import Image
import io, base64
from pathlib import Path
from tqdm import tqdm

dataset_dir = Path('dataset_images')
dataset_dir.mkdir(exist_ok=True)
dataset_samples = []

# Supported 13 languages
LANGUAGES = ['en', 'he', 'ar', 'es', 'fr', 'de', 'it', 'pt', 'ru', 'zh', 'ja', 'hi', 'nl']

# Canonical System Prompt from DressApp
PROMPT_GARMENT_VISION = (
    'Output raw JSON only ({...} or [{...}]). No markdown/intro.\n'
    '• name & title: Short, unique (2-4 words) extracting cut/attributes (e.g. \'Green Round-Toe Loafers\', \'Navy Chinos\', \'Camo Cargo Pants\'). Never generic (\'Green Garment\', \'Clothing\').\n'
    '• sub_category: Specific cut (\'T-Shirt\',\'Sweater\',\'Jeans\',\'Pants\',\'Skirt\',\'Shoes\',\'Sneakers\',\'Sandals\',\'Boots\',\'Loafers\',\'Sunglasses\',\'Handbag\'). Never \'Top\'/\'Bottom\'/\'Footwear\'.\n'
    '• Bottoms: \'Jeans\'=denim. Chinos -> sub_category:\'Pants\', item_type:\'Chinos\', dress_code:\'smart-casual\'. Sweatpants/joggers -> sub_category:\'Pants\', item_type:\'Sweatpants\'|\'Joggers\', dress_code:\'casual\'|\'athletic\'.\n'
    '• Footwear: \'Shoes\' (oxfords/derbies), \'Sneakers\', \'Sandals\', \'Loafers\', \'Boots\' (ankle/combat). Laced/ankle are \'Boots\'; low dress shoes are \'Shoes\'.\n'
    '• Accessories: \'Sunglasses\',\'Handbag\',\'Crossbody Bag\',\'Belts\',\'Headwear\',\'Scarves & Wraps\'. Non-wearables -> is_clothing:false.\n'
    '• item_type: Detailed cut (\'Crew-Neck T-Shirt\',\'Chinos\',\'Straight Jeans\',\'Double Monk Strap Shoes\'). Differ from sub_category.\n'
    '• caption: <=12 words fluent sentence in requested language. End with period. No brackets/lists.\n'
    '• dress_code: \'casual\'|\'smart-casual\'|\'business\'|\'formal\'|\'athletic\'|\'loungewear\'. Suits=\'business\'; shirts=\'smart-casual\'; tuxedos=\'formal\'.\n'
    '• model_gender: Model -> \'women\'|\'men\'. Flat lay/hanger/mannequin -> null.\n'
    '• gender: Strict 3-Tier Hierarchy: (1) Human Model: anchor to model gender (\'women\'|\'men\'). (2) Garment Criteria: flat lays determined by cut (\'women\' for floral/blouses/skirts/sandals; \'men\' for masculine cuts; \'unisex\' for neutral basics). (3) Neutral basics fall back to profile gender, or \'unisex\'. Never default to \'men\'.\n'
    '• colors: ALWAYS [{\"name\": str, \"pct\": int}] summing to 100. Indoor white/cream tops must be \'white\'/\'cream\', never \'grey\'.\n'
    '• fabric_materials: [{\"name\": str, \"pct\": int}] summing to 100. tags: [str] (3-4 unique tags, never duplicate sub_category).\n'
    '• pattern: \'solid\'|\'printed\'|\'geometric\'|\'striped\'|\'plaid\'|\'floral\'|\'camouflage\'.\n'
    '• text/logos: Read accurately (\'American Eagle\'=eagle/עיט, not deer/אייל).\n'
    '• season: [\'spring\'|\'summer\'|\'fall\'|\'winter\']. Basics=[\'spring\',\'summer\',\'fall\',\'winter\'].\n'
    '• Quality & Repair: Only emit \'reconstruction_prompt\' if image_quality_status != \'complete\'. Set null if complete.'
)

PROMPT_SCHEDULED_OUTFIT = (
    'You are DressApp\'s Automated Outfit Scheduler.\n'
    'Generate scheduled, weather-ready outfit proposals from the user\'s closet.\n\n'
    'RULES:\n'
    '• Complete Outfits: Every look MUST have: 1) top+bottom OR dress, 2) footwear/shoes. Never single items.\n'
    '• Versatility: Mix and match items across dates while avoiding consecutive identical pieces.\n'
    '• Context Matching: Adapt strictly to forecast weather, temperature, and calendar events.\n'
    '• Anatomical Roles: \'top\'|\'bottom\'|\'outerwear\'|\'shoes\'|\'accessory\'|\'dress\'.\n'
    '• Output contract: Return ONLY a JSON object: {\"proposals\": Array<{\"date\": string, \"title\": string, \"items\": Array<{\"role\": string, \"closet_item_id\": string, \"name\": string}>, \"vibe\": string, \"weather_rationale\": string}>}'
)

# Authentic User Prompts for all 13 supported languages (exact match with backend/app/services/vision/llm.py)
def get_user_prompt(lang_code):
    prompts = {
        'en': "Analyze photo. Return raw JSON (1 object or array). No commentary.\n• name & title: Short, unique, descriptive garment name (2-5 words) extracting the specific cut and key attributes (e.g. 'Green Round-Toe Loafers', 'Brown Camouflage Cargo Pants', 'Brown Leather Oxford Shoes', 'Olive Pleated Skirt', 'Navy Chinos', 'White Linen Button-Down Shirt'). NEVER generic like 'Green Garment', 'Piece of clothing', or 'Clothing'.\n• sub_category != item_type; pattern: 'camouflage'|'printed'|'floral'|'geometric'|'solid'; dress_code: formality; gender: criteria ('women' for floral/feminine/blouses, 'men' for masculine cuts, 'unisex' for neutral; if uncertain, apply 'unisex'; never default to 'men').",
        'he': "**OUTPUT LANGUAGE: Hebrew (עברית)**\nAll strings in fluent modern Hebrew (חולצת טי, ג'ינס). Keys and enum values in English.\n• name & title: שם ייחודי, תמציתי ומדויק (2-5 מילים) הכולל גזרה ומאפיינים (למשל 'נעלי אוקספורד חומות', 'מוקסינים ירוקים', 'מכנסי דגמ\"ח הסוואה', 'חצאית פליסה ירוקה', 'מכנסי צ'ינו זית'). לעולם אל תשתמש בשמות כלליים כמו 'בגד ירוק', 'בגד חום' או 'בגד'.\n• sub_category != item_type; הסוואה=pattern:'camouflage'; פרחוני=pattern:'floral'; dress_code: רשמיות; מגדר: נתח גזרה והדפס ('women' לפרחוני/נשי/בלוזות, 'men' לגברי, 'unisex' לפריטים ניטרליים; אם לא ודאי, 'unisex'; לעולם אל תניח אוטומטית 'men').\n• Graphic text: 'AMERICAN EAGLE'=עיט/נשר (not deer). Return raw JSON.",
        'ar': "**OUTPUT LANGUAGE: Arabic (العربية)**\nAll strings in fluent Arabic. Keys and enum values in English.\n• name & title: اسم فريد وموجز ومحدد للقطعة (2-5 كلمات) يتضمن القصة والصفات الأساسية (مثل 'حذاء لوفر أخضر مدور', 'حذاء أكسفورد جلدي بني', 'تنورة بليسيه خضراء', 'بنطال تشينو كحلي'). تجنب تماماً الأسماء العامة مثل 'ملابس خضراء' أو 'قطعة ملابس'.\n• sub_category != item_type; dress_code: formality; gender: criteria ('women' for floral/feminine, 'men' for masculine, 'unisex' for neutral; إذا كان غير مؤكد، 'unisex'; never default to 'men').\n• Return raw JSON.",
        'es': "**OUTPUT LANGUAGE: Spanish (Español)**\nAll strings in fluent Spanish. Keys and enum values in English.\n• name & title: Nombre descriptivo, único y conciso (2-5 palabras) que extrae el corte y atributos clave (ej. 'Mocasines verdes de punta redonda', 'Zapatos Oxford de cuero marrón', 'Falda plisada verde oliva', 'Pantalones chinos azul marino'). NUNCA uses nombres genéricos como 'Prenda verde' o 'Ropa'.\n• sub_category != item_type; dress_code: formalidad; gender: criterios ('women' para floral/femenino, 'men' para corte masculino, 'unisex' para neutro; si no está seguro, 'unisex'; nunca asumas 'men' por defecto).\n• Return raw JSON.",
        'fr': "**OUTPUT LANGUAGE: French (Français)**\nAll strings in fluent French. Keys and enum values in English.\n• name & title: Nom descriptif, unique et précis (2-5 mots) extrayant la coupe et les attributs clés (ex. 'Mocassins verts à bout rond', 'Chaussures richelieu en cuir marron', 'Jupe plissée olive', 'Pantalon chino bleu marine'). Ne JAMAIS utiliser de noms génériques comme 'Vêtement vert' ou 'Vêtement'.\n• sub_category != item_type; dress_code: formalité; gender: critères ('women' pour floral/féminin, 'men' pour coupe masculine, 'unisex' pour neutre; si incertain, 'unisex'; ne jamais choisir 'men' par défaut).\n• Return raw JSON.",
        'de': "**OUTPUT LANGUAGE: German (Deutsch)**\nAll strings in fluent German. Keys and enum values in English.\n• name & title: Kurzer, eindeutiger, beschreibender Kleidungsname (2-5 Wörter) mit Schnitt und Hauptmerkmalen (z. B. 'Grüne Loafer mit runder Spitze', 'Braune Leder-Oxfordschuhe', 'Olivgrüner Plisseerock', 'Marineblaue Chinos'). NIEMALS generische Namen wie 'Grünes Kleidungsstück' oder 'Kleidung'.\n• sub_category != item_type; dress_code: Formalität; gender: Kriterien ('women' für floral/feminin, 'men' für maskuline Schnitte, 'unisex' für neutrale Basics; bei Unsicherheit 'unisex'; niemals standardmäßig 'men').\n• Return raw JSON.",
        'it': "**OUTPUT LANGUAGE: Italian (Italiano)**\nAll strings in fluent Italian. Keys and enum values in English.\n• name & title: Nome descrittivo, unico e conciso (2-5 parole) che specifica taglio e caratteristiche chiave (es. 'Mocassini verdi a punta tonda', 'Scarpe Oxford in pelle marrone', 'Gonna a pieghe verde oliva', 'Pantaloni chino blu navy'). MAI nomi generici come 'Capo verde' o 'Vestito'.\n• sub_category != item_type; dress_code: formalità; gender: criteri ('women' per floreale/femminile, 'men' per tagli maschili, 'unisex' per neutro; se incerto, 'unisex'; mai impostare 'men' come predefinito).\n• Return raw JSON.",
        'pt': "**OUTPUT LANGUAGE: Portuguese (Português)**\nAll strings in fluent Portuguese. Keys and enum values in English.\n• name & title: Nome descritivo, exclusivo e conciso (2-5 palavras) destacando corte e atributos principais (ex. 'Mocassins verdes de bico redondo', 'Sapatos Oxford de couro marrom', 'Saia plissada oliva', 'Calça chino azul-marinho'). NUNCA use nomes genéricos como 'Peça verde' ou 'Roupa'.\n• sub_category != item_type; dress_code: formalidade; gender: critérios ('women' para floral/feminino, 'men' para cortes masculinos, 'unisex' para neutro; se incerto, 'unisex'; nunca definir 'men' por padrão).\n• Return raw JSON.",
        'ru': "**OUTPUT LANGUAGE: Russian (Русский)**\nAll strings in fluent Russian. Keys and enum values in English.\n• name & title: Короткое, уникальное, описательное название вещи (2-5 слов) с указанием фасона и ключевых деталей (напр., 'Зеленые лоферы с круглым носком', 'Коричневые кожаные оксфорды', 'Оливковая юбка-плиссе', 'Темно-синие брюки чинос'). НИКОГДА не используйте общие названия вроде 'Зеленая вещь' или 'Одежда'.\n• sub_category != item_type; dress_code: формальность; gender: критерии ('women' для женственного/цветочного, 'men' для мужского кроя, 'unisex' для нейтрального; если неуверенно, 'unisex'; никогда не ставить 'men' по умолчанию).\n• Return raw JSON.",
        'zh': "**OUTPUT LANGUAGE: Chinese Simplified (简体中文)**\nAll strings in fluent Simplified Chinese. Keys and enum values in English.\n• name & title: 简短、独特且具体的服装名称（2-5个词/字），准确提取剪裁与核心属性（例如：'绿色圆头乐福鞋'、'棕色真皮牛津鞋'、'橄榄绿百褶裙'、'藏青色斜纹棉裤'、'白色亚麻衬衫'）。严禁使用笼统名称如'绿色衣服'、'单品'或'服装'。\n• sub_category != item_type; dress_code: 着装规范; gender: 判别标准（碎花/女性剪裁为'women'，男士剪裁为'men'，中性基础款为'unisex'；若不确定，'unisex'；绝不默认填'men'）。\n• Return raw JSON.",
        'ja': "**OUTPUT LANGUAGE: Japanese (日本語)**\nAll strings in fluent Japanese. Keys and enum values in English.\n• name & title: カッティングや主要な特徴を反映した、簡潔で固有の説明的なアイテム名（2〜5語）（例：「グリーンのラウンドトゥローファー」「ブラウンレザーのオックスフォードシューズ」「オリーブプリーツスカート」「ネイビーチノパン」）。「緑の服」や「衣類」のような一般的な表現は絶対に避けてください。\n• sub_category != item_type; dress_code: フォーマリティ; gender: 基準（花柄・フェミニンは'women'、メンズカットは'men'、ニュートラルは'unisex'；不明な場合は'unisex'；デフォルトで'men'にしないこと）。\n• Return raw JSON.",
        'hi': "**OUTPUT LANGUAGE: Hindi (हिन्दी)**\nAll strings in fluent Hindi. Keys and enum values in English.\n• name & title: विशिष्ट, संक्षिप्त और सटीक परिधान नाम (2-5 शब्द) जिसमें कट और मुख्य विशेषताएं शामिल हों (जैसे 'हरे गोल पैर के लोफर्स', 'भूरे चमड़े के ऑक्सफ़ोर्ड जूते', 'जैतून प्लीटेड स्कर्ट', 'नेवी चिनो पैंट')। कभी भी 'हरा कपड़ा' या 'कपड़े' जैसे सामान्य नाम न दें।\n• sub_category != item_type; dress_code: औपचारिकता; gender: मानदंड (फ्लोरल/स्त्री कट के लिए 'women', पुरुष कट के लिए 'men', तटस्थ के लिए 'unisex'; यदि अनिश्चित हो तो 'unisex'; कभी भी डिफ़ॉल्ट रूप से 'men' न चुनें)।\n• Return raw JSON.",
        'nl': "**OUTPUT LANGUAGE: Dutch (Nederlands)**\nAll strings in fluent Dutch. Keys and enum values in English.\n• name & title: Korte, unieke, beschrijvende kledingnaam (2-5 woorden) met het specifieke model en kenmerken (bijv. 'Groene ronde neus loafers', 'Bruine leren oxford schoenen', 'Olijfgroene plooirok', 'Donkerblauwe chino'). Nooit generieke namen zoals 'Groen kledingstuk' of 'Kleding'.\n• sub_category != item_type; dress_code: formaliteit; gender: criteria ('women' voor floraal/vrouwelijk, 'men' voor mannelijke snit, 'unisex' voor neutrale basics; indien onzeker, 'unisex'; nooit standaard 'men').\n• Return raw JSON.",
    }
    return prompts.get(lang_code, prompts['en'])

# Multilingual Fashion Vocabulary Dictionaries (13 Languages)
        TRANSLATIONS = {
            'colors': {
                'black': {'en':'Black','he':'שחור','ar':'أسود','es':'Negro','fr':'Noir','de':'Schwarz','it':'Nero','pt':'Preto','ru':'Черный','zh':'黑色','ja':'ブラック','hi':'काला','nl':'Zwart'},
                'white': {'en':'White','he':'לבן','ar':'أبيض','es':'Blanco','fr':'Blanc','de':'Weiß','it':'Bianco','pt':'Branco','ru':'Белый','zh':'白色','ja':'ホワイト','hi':'सफेद','nl':'Wit'},
                'blue': {'en':'Blue','he':'כחול','ar':'أزرق','es':'Azul','fr':'Bleu','de':'Blau','it':'Blu','pt':'Azul','ru':'Синий','zh':'蓝色','ja':'ブルー','hi':'नीला','nl':'Blauw'},
                'navy': {'en':'Navy Blue','he':'כחול נייבי','ar':'كحلي داكن','es':'Azul marino','fr':'Bleu marine','de':'Marineblau','it':'Blu navy','pt':'Azul-marinho','ru':'Темно-синий','zh':'藏青色','ja':'ネイビー','hi':'नेवी ब्लू','nl':'Marineblauw'},
                'brown': {'en':'Brown','he':'חום','ar':'بني','es':'Marrón','fr':'Marron','de':'Braun','it':'Marrone','pt':'Marrom','ru':'Коричневый','zh':'棕色','ja':'ブラウン','hi':'भूरा','nl':'Bruin'},
                'green': {'en':'Green','he':'ירוק','ar':'أخضر','es':'Verde','fr':'Vert','de':'Grün','it':'Verde','pt':'Verde','ru':'Зеленый','zh':'绿色','ja':'グリーン','hi':'हरा','nl':'Groen'},
                'olive': {'en':'Olive Green','he':'ירוק זית','ar':'زيتي','es':'Verde oliva','fr':'Olive','de':'Olivgrün','it':'Verde oliva','pt':'Verde-oliva','ru':'Оливковый','zh':'橄榄绿','ja':'オリーブ','hi':'जैतून हरा','nl':'Olijfgroen'},
                'red': {'en':'Red','he':'אדום','ar':'أحمر','es':'Rojo','fr':'Rouge','de':'Rot','it':'Rosso','pt':'Vermelho','ru':'Красный','zh':'红色','ja':'レッド','hi':'लाल','nl':'Rood'},
                'yellow': {'en':'Yellow','he':'צהוב','ar':'أصفر','es':'Amarillo','fr':'Jaune','de':'Gelb','it':'Giallo','pt':'Amarelo','ru':'Желтый','zh':'黄色','ja':'イエロー','hi':'पीला','nl':'Geel'},
                'pink': {'en':'Pink','he':'ורוד','ar':'وردي','es':'Rosa','fr':'Rose','de':'Rosa','it':'Rosa','pt':'Rosa','ru':'Розовый','zh':'粉色','ja':'ピンク','hi':'गुलाबी','nl':'Roze'},
                'grey': {'en':'Charcoal Grey','he':'אפור פחם','ar':'رمادي فحمي','es':'Gris marengo','fr':'Gris anthracite','de':'Anthrazitgrau','it':'Grigio antracite','pt':'Cinza chumbo','ru':'Темно-серый','zh':'炭灰色','ja':'チャコールグレー','hi':'धूसर','nl':'Antracietgrijs'},
                'beige': {'en':'Warm Beige','he':'בז\' בהיר','ar':'بيج دافئ','es':'Beige cálido','fr':'Beige clair','de':'Helles Beige','it':'Beige caldo','pt':'Bege suave','ru':'Светло-бежевый','zh':'米白色','ja':'ライトベージュ','hi':'हल्का बेज','nl':'Warm beige'},
                'khaki': {'en':'Khaki','he':'חאקי','ar':'كاكي','es':'Caqui','fr':'Kaki','de':'Khaki','it':'Kaki','pt':'Cáqui','ru':'Хаки','zh':'卡其色','ja':'カーキ','hi':'खाकी','nl':'Khaki'},
            },
            'dress_code': {
                'casual': {
                    'en': 'casual', 'he': 'יומיומי', 'ar': 'يومي', 'es': 'casual', 'fr': 'décontracté',
                    'de': 'lässigen', 'it': 'casual', 'pt': 'casual', 'ru': 'повседневного', 'zh': '日常休闲',
                    'ja': 'カジュアル', 'hi': 'कैज़ुअल', 'nl': 'casual'
                },
                'smart-casual': {
                    'en': 'smart-casual', 'he': 'אלגנטי', 'ar': 'أنيق وعصري', 'es': 'elegante casual', 'fr': 'chic décontracté',
                    'de': 'eleganten Casual', 'it': 'smart casual', 'pt': 'casual elegante', 'ru': 'смарт-кэжуал', 'zh': '优雅轻商务',
                    'ja': 'スマートカジュアル', 'hi': 'स्मार्ट कैज़ुअल', 'nl': 'smart casual'
                },
                'formal': {
                    'en': 'formal', 'he': 'רשמי', 'ar': 'رسمي', 'es': 'formal', 'fr': 'habillé',
                    'de': 'formellen', 'it': 'formale', 'pt': 'formal', 'ru': 'делового', 'zh': '正式职场',
                    'ja': 'フォーマル', 'hi': 'औपचारिक', 'nl': 'formeel'
                },
                'athletic': {
                    'en': 'active athletic', 'he': 'ספורטיבי ופעיל', 'ar': 'رياضي نشط', 'es': 'deportivo activo', 'fr': 'sportif',
                    'de': 'sportlichen', 'it': 'sportivo', 'pt': 'esportivo', 'ru': 'спортивного', 'zh': '活力运动',
                    'ja': 'スポーツ', 'hi': 'स्पोर्टी एक्टिव', 'nl': 'sportief'
                },
                'business': {
                    'en': 'business professional', 'he': 'עסקי ייצוגי', 'ar': 'عمل رسمي', 'es': 'de negocios', 'fr': 'professionnel',
                    'de': 'Business', 'it': 'business', 'pt': 'corporativo', 'ru': 'делового', 'zh': '商务正装',
                    'ja': 'ビジネス', 'hi': 'व्यावसायिक', 'nl': 'zakelijk'
                },
                'loungewear': {
                    'en': 'relaxed lounge', 'he': 'נוח וביתי', 'ar': 'مريح منزلي', 'es': 'cómodo', 'fr': 'détente',
                    'de': 'gemütlichen Lounge', 'it': 'relax', 'pt': 'confortável', 'ru': 'домашнего', 'zh': '居家休闲',
                    'ja': 'リラックス', 'hi': 'आरामदेह लाउंज', 'nl': 'comfortabel'
                },
            },
            'items': {
                'High Heel Pumps': {'en':'High Heel Pumps','he':'נעלי עקב קלאסיות','ar':'حذاء كعب عالي كلاسيكي','es':'Zapatos de tacón alto','fr':'Escarpins à talons hauts','de':'High-Heel-Pumps','it':'Décolleté con tacco alto','pt':'Sapatos de salto alto','ru':'Классические туфли на каблуке','zh':'优雅高跟单鞋','ja':'ハイヒールパンプス','hi':'ऊँची एड़ी के जूते','nl':'Hoge hak pumps'},
                'Stiletto Heels': {'en':'Stiletto Heels','he':'נעלי עקב סטילטו אלגנטיות','ar':'حذاء ستيلتو بكعب رفيع','es':'Tacones de aguja stiletto','fr':'Talons aiguilles stiletto','de':'Stiletto-Pumps','it':'Tacchi a spillo stiletto','pt':'Saltos agulha stiletto','ru':'Туфли на шпильке','zh':'精致细高跟鞋','ja':'スティレットヒール','hi':'स्टिलेट्टो हील्स','nl':'Stiletto hakken'},
                'Heeled Pumps': {'en':'Heeled Pumps','he':'נעלי עקב מעוצבות','ar':'حذاء كعب أنيق','es':'Zapatos de tacón','fr':'Escarpins classiques','de':'Klassische Pumps','it':'Scarpe décolleté','pt':'Sapatos de salto','ru':'Туфли на среднем каблуке','zh':'通勤中跟单鞋','ja':'パンプス','hi':'हील पंप्स','nl':'Pumps met hak'},
                'Ballet Flats': {'en':'Ballet Flats','he':'נעלי בלרינה שטוחות','ar':'حذاء باليه مسطח','es':'Bailarinas planas','fr':'Ballerines plates','de':'Ballet-Ballerinas','it':'Ballerine basse','pt':'Sapatilhas femininas','ru':'Классические балетки','zh':'舒适芭蕾平底鞋','ja':'バレエシューズ','hi':'बैले फ्लैट्स','nl':'Ballerina flats'},
                'Leather Loafers': {'en':'Leather Loafers','he':'מוקסינים מעור איכותי','ar':'حذاء لوفر جلدي فاخر','es':'Mocasines de cuero','fr':'Mocassins en cuir','de':'Leder-Loafer','it':'Mocassini in pelle','pt':'Mocassins de couro','ru':'Кожаные лоферы','zh':'真皮乐福鞋','ja':'レザーローファー','hi':'चमड़े के लोफर्स','nl':'Leren loafers'},
                'Oxford Dress Shoes': {'en':'Oxford Dress Shoes','he':'נעלי אוקספורד אלגנטיות','ar':'حذاء أكسفورد رسمي كلاسيكي','es':'Zapatos Oxford elegantes','fr':'Chaussures richelieu habillées','de':'Elegante Oxford-Schuhe','it':'Scarpe Oxford eleganti','pt':'Sapatos sociais Oxford','ru':'Элегантные кожаные оксфорды','zh':'经典商务牛津鞋','ja':'ドレスオックスフォード','hi':'ऑक्सफ़ोर्ड फॉर्मल जूते','nl':'Nette oxford schoenen'},
                'Casual Slip-Ons': {'en':'Casual Slip-Ons','he':'נעלי סליפ-און נוחות','ar':'حذاء كاجوال سهل الارتداء','es':'Zapatillas slip-on casuales','fr':'Chaussures slip-on décontractées','de':'Bequeme Slip-On Schuhe','it':'Scarpe slip-on casual','pt':'Calçados slip-on casuais','ru':'Повседневные слипоны','zh':'休闲一脚蹬单鞋','ja':'スリッポンシューズ','hi':'आरामदायक स्लिप-ऑन जूते','nl':'Casual instappers'},
                'Running Sneakers': {'en':'Running Sneakers','he':'נעלי סניקרס ספורטיביות','ar':'حذاء رياضي مريح للجري','es':'Zapatillas deportivas de running','fr':'Baskets de course légères','de':'Leichte Lauf-Sneaker','it':'Sneakers sportive da corsa','pt':'Tênis de corrida leve','ru':'Спортивные кроссовки для бега','zh':'轻便跑步运动鞋','ja':'ランニングスニーカー','hi':'रनिंग स्नीकर्स','nl':'Hardloopsneakers'},
                'Leather Boots': {'en':'Leather Boots','he':'מגפי עור גבוהים','ar':'حذاء بوت جلدي متين','es':'Botas de cuero clásicas','fr':'Bottes en cuir élégantes','de':'Klassische Lederstiefel','it':'Stivali in pelle alti','pt':'Botas de couro elegantes','ru':'Высокие кожаные сапоги','zh':'经典真皮长靴','ja':'レザーブーツ','hi':'चमड़े के जूते','nl':'Leren laarzen'},
                'Ankle Boots': {'en':'Ankle Boots','he':'מגפוני קרסול אופנתיים','ar':'حذاء كاحل بوت أنيق','es':'Botines de tobillo modernos','fr':'Bottines cheville élégantes','de':'Modische Knöchelstiefel','it':'Stivaletti alla caviglia','pt':'Botinas de cano curto','ru':'Элегантные ботильоны','zh':'时尚及踝短靴','ja':'アンクルブーツ','hi':'टखने के बूट्स','nl':'Enkellaarsjes'},
                'Strappy Sandals': {'en':'Strappy Sandals','he':'סנדלי רצועות קיציים','ar':'صندل صيفي بأشرطة','es':'Sandalias veraniegas de tiras','fr':'Sandales d\'été à brides','de':'Riemchensandalen','it':'Sandali estivi con cinturini','pt':'Sandálias de tiras de verão','ru':'Летние босоножки с ремешками','zh':'清凉细带凉鞋','ja':'ストラップサンダル','hi':'स्ट्रैपी सैंडल','nl':'Sandalen met bandjes'},
                'A-Line Skirt': {'en':'A-Line Skirt','he':'חצאית מעוצבת בגזרת A','ar':'تنورة أنيقة بقصة A','es':'Falda acampanada línea A','fr':'Jupe trapèze élégante','de':'Klassischer A-Linien-Rock','it':'Gonna a trapezio','pt':'Saia evasê moderna','ru':'Юбка А-силуэта','zh':'修身A字半身裙','ja':'Aラインスカート','hi':'ए-लाइन स्कर्ट','nl':'A-lijn rok'},
                'Maxi Skirt': {'en':'Maxi Skirt','he':'חצאית מקסי ארוכה וזורמת','ar':'تنورة ماكسي طويلة وأنيقة','es':'Falda maxi larga y fluida','fr':'Jupe longue fluide maxi','de':'Langer fließender Maxirock','it':'Gonna lunga maxi fluida','pt':'Saia longa maxi fluida','ru':'Длинная струящаяся юбка макси','zh':'优雅飘逸长裙','ja':'マキシスカート','hi':'मैक्सी स्कर्ट','nl':'Maxi-rok'},
                'Pleated Skirt': {'en':'Pleated Skirt','he':'חצאית פליסה אלגנטית','ar':'تنورة بليسيه مثنية أنيقة','es':'Falda plisada elegante','fr':'Jupe plissée raffinée','de':'Eleganter Plisseerock','it':'Gonna a pieghe elegante','pt':'Saia plissada moderna','ru':'Элегантная юбка-плиссе','zh':'百褶风琴半身裙','ja':'プリーツスカート','hi':'प्लीटेड स्कर्ट','nl':'Plooirok'},
                'Tailored Chinos': {'en':'Tailored Chinos','he':'מכנסי צ\'ינו מחויטים','ar':'بنطال تشينو كلاسيكي أنيق','es':'Pantalones chinos a medida','fr':'Pantalon chino ajusté','de':'Klassische Chino-Hose','it':'Pantaloni chino su misura','pt':'Calça chino alinhada','ru':'Классические брюки чинос','zh':'修身纯棉斜纹裤','ja':'テーラードチノパン','hi':'चिनो पैंट','nl':'Nette chino broek'},
                'Tailored Trousers': {'en':'Tailored Trousers','he':'מכנסיים מחויטים קלאסיים','ar':'بنطال رسمي أنيق ومتقן','es':'Pantalones de vestir a medida','fr':'Pantalon de costume habillé','de':'Elegante Stoffhose','it':'Pantaloni sartoriali classici','pt':'Calça social de alfaiataria','ru':'Классические костюмные брюки','zh':'经典正装西裤','ja':'スラックス','hi':'फॉर्मल पैंट','nl':'Nette pantalon'},
                'Slim-Fit Denim Jeans': {'en':'Slim-Fit Denim Jeans','he':'ג\'ינס סלים-פיט איכותי','ar':'بنطال جينز ضيق عصري','es':'Vaqueros denim de corte slim','fr':'Jean slim en denim moderne','de':'Slim-Fit Denim Jeans','it':'Jeans slim in denim elasticizzato','pt':'Calça jeans slim moderna','ru':'Джинсы слим из денима','zh':'修身微弹牛仔裤','ja':'スリムフィットジーンズ','hi':'स्लिम-फिट जींस','nl':'Slim-fit spijkerbroek'},
                'Crewneck Cotton T-Shirt': {'en':'Crewneck Cotton T-Shirt','he':'חולצת טי כותנה קלאסית','ar':'تي شيرت قطني بياقة مستديرة','es':'Camiseta de algodón de cuello redondo','fr':'T-shirt en coton à col rond','de':'Klassisches Rundhals-Baumwoll-T-Shirt','it':'T-shirt in cotone girocollo','pt':'Camiseta de algodão gola careca','ru':'Хлопковая футболка с круглым вырезом','zh':'经典圆领纯棉T恤','ja':'クルーネックコットンTシャツ','hi':'सूती गोल गले की टी-शर्ट','nl':'Katoenen T-shirt met ronde hals'},
                'Button-Down Shirt': {'en':'Button-Down Shirt','he':'חולצה מכופתרת קלאסית','ar':'قميص بأزرار وياقة كلاسيكية','es':'Camisa clásica con botones','fr':'Chemise boutonnée classique','de':'Klassisches Button-Down-Hemd','it':'Camicia abbottonata classica','pt':'Camisa social de botões','ru':'Классическая рубашка на пуговицах','zh':'长袖纽扣商务衬衫','ja':'ボタンダウンシャツ','hi':'बटन-डाउन शर्ट','nl':'Overhemd met knopen'},
                'A-Line Casual Dress': {'en':'A-Line Casual Dress','he':'שמלת יומיום בגזרת A','ar':'فستان كاجوال أنيق بقصة A','es':'Vestido casual en corte A','fr':'Robe trapèze décontractée','de':'Lässiges A-Linien-Kleid','it':'Abito casual a trapezio','pt':'Vestido casual evasê','ru':'Повседневное платье А-силуэта','zh':'休闲A字连衣裙','ja':'Aラインカジュアルワンピース','hi':'ए-लाइन कैज़ुअल ड्रेस','nl':'Casual A-lijn jurk'},
                'Maxi Dress': {'en':'Maxi Dress','he':'שמלת מקסי קיצית ארוכה','ar':'فستان ماكسي طويل وأنيق','es':'Vestido maxi largo elegante','fr':'Robe longue maxi raffinée','de':'Elegantes langes Maxikleid','it':'Abito lungo maxi elegante','pt':'Vestido longo maxi refinado','ru':'Длинное женственное платье макси','zh':'浪漫大摆长款连衣裙','ja':'マキシワンピース','hi':'मैक्सी ड्रेस','nl':'Maxi-jurk'},
                'Tailored Blazer': {'en':'Tailored Blazer','he':'בלייזר מחויט ואלגנטי','ar':'سترة بليزر رسمية أنيقة','es':'Blazer entallado elegante','fr':'Blazer ajusté raffiné','de':'Taillierter eleganter Blazer','it':'Blazer sartoriale elegante','pt':'Blazer estruturado alinhado','ru':'Элегантный приталенный блейзер','zh':'修身剪裁西装外套','ja':'テーラードブレザー','hi':'टेलर ब्लेज़र','nl':'Getailleerde blazer'},
                'Structured Jacket': {'en':'Structured Jacket','he':'ז\'קט מובנה ומעוצב','ar':'جاكيت عصري منظم','es':'Chaqueta estructurada moderna','fr':'Veste structurée moderne','de':'Moderne strukturierte Jacke','it':'Giacca strutturata moderna','pt':'Jaqueta estruturada moderna','ru':'Современная структурированная куртка','zh':'立体剪裁夹克外套','ja':'ジャケット','hi':'स्ट्रक्चर्ड जैकेट','nl':'Gestructureerd jack'},
                'Cargo Pants': {'en':'Cargo Pants','he':'מכנסי דגמ\"ח מעוצבים','ar':'بنطال كارغو بجيوب متعددة','es':'Pantalones cargo con bolsillos','fr':'Pantalon cargo multipoches','de':'Praktische Cargohose','it':'Pantaloni cargo con tasconi','pt':'Calça cargo com bolsos','ru':'Практичные брюки карго','zh':'多口袋工装裤','ja':'カーゴパンツ','hi':'कार्गो पैंट','nl':'Cargobroek'},
                'Silk Blouse': {'en':'Silk Blouse','he':'חולצת משי רכה ואלגנטית','ar':'بلوزة حريرية ناعمة وفاخرة','es':'Blusa de seda suave y elegante','fr':'Blouse en soie douce et raffinée','de':'Elegante Seidenbluse','it':'Camicetta in seta morbida ed elegante','pt':'Blusa de seda suave e refinada','ru':'Шелковая струящаяся блузка','zh':'优雅真丝柔滑衬衫','ja':'シルクブラウス','hi':'सिल्क ब्लाउज','nl':'Zijden blouse'},
                'Cotton Shirt': {'en':'Cotton Shirt','he':'חולצת כותנה נושמת','ar':'قميص قطني مريح','es':'Camisa de algodón transpirable','fr':'Chemise en coton respirant','de':'Atmungsaktives Baumwollhemd','it':'Camicia in cotone traspirante','pt':'Camisa de algodão leve','ru':'Хлопковая комфортная рубашка','zh':'透气纯棉休闲衬衫','ja':'コットンシャツ','hi':'सूती शर्ट','nl':'Katoenen overhemd'},
                'Leather Handbag': {'en':'Leather Handbag','he':'תיק יד מעור איכותי','ar':'حقيبة يد جلدية فاخرة','es':'Bolso de mano de cuero','fr':'Sac à main en cuir élégant','de':'Elegante Lederhandtasche','it':'Borsa a mano in pelle raffinata','pt':'Bolsa de mão de couro','ru':'Элегантная кожаная сумка','zh':'真皮手提包','ja':'レザーハンドバッグ','hi':'चमड़े का हैंडबैग','nl':'Leren handtas'},
                'Crossbody Bag': {'en':'Crossbody Bag','he':'תיק צד קומפקטי','ar':'حقيبة كروسبودي عملية','es':'Bolso bandolera práctico','fr':'Sac bandoulière compact','de':'Kompakte Umhängetasche','it':'Borsa a tracolla compatta','pt':'Bolsa transversal prática','ru':'Компактная сумка через плечо','zh':'百搭斜挎包','ja':'クロスボディバッグ','hi':'क्रॉसबॉडी बैग','nl':'Crossbody tas'},
                'Canvas Tote Bag': {'en':'Canvas Tote Bag','he':'תיק טוט מבד קנבס','ar':'حقيبة توت قماشية واسعة','es':'Bolso tote de lona resistente','fr':'Sac cabas en toile résistant','de':'Geräumige Canvas-Totetasche','it':'Borsa tote in tela resistente','pt':'Bolsa tote de lona espaçosa','ru':'Вместительная сумка-тоут из канваса','zh':'实用帆布托特包','ja':'キャンバストートバッグ','hi':'कैनवास टोट बैग','nl':'Canvas tote bag'},
            },
            'care': {
                'footwear': {
                    'en': 'Wipe clean with a soft damp cloth and store with shoe trees to maintain shape.',
                    'he': 'יש לנקות במטלית לחה ורכה ולאחסן עם אימום לשמירה על צורת הנעל.',
                    'ar': 'امسح بقطعة قماش ناعمة ورطبة واحفظه مع قوالب الأحذية للحفاظ على الشكل.',
                    'es': 'Limpiar con un paño suave húmedo y guardar con hormas para mantener la forma.',
                    'fr': 'Nettoyer avec un chiffon doux humide et ranger avec des embauchoirs.',
                    'de': 'Mit feuchtem Tuch abwischen und mit Schuhspannern lagern.',
                    'it': 'Pulire con un panno morbido umido e riporre con tendiscarpe per mantenere la forma.',
                    'pt': 'Limpar com pano macio úmido e guardar com formas para sapatos.',
                    'ru': 'Протирать мягкой влажной тканью и хранить с колодками для формы.',
                    'zh': '用柔软湿布擦拭干净，使用鞋撑定型收纳。',
                    'ja': '柔らかい湿った布で拭き取り、シューキーパーを入れて保管してください。',
                    'hi': 'मुलायम नम कपड़े से पोंछें और आकार बनाए रखने के लिए शू ट्री के साथ रखें।',
                    'nl': 'Afnemen met een zachte vochtige doek en bewaren met schoenspanners.',
                },
                'apparel': {
                    'en': 'Machine wash cold with like colors inside out; tumble dry low or hang dry.',
                    'he': 'כביסה עדינה במכונה במים קרים עם צבעים דומים הפוכים; ייבוש בתלייה.',
                    'ar': 'يغسل في الغسالة بماء بارد مع ألوان مماثلة مقلوباً؛ يجفف بالتعليق.',
                    'es': 'Lavar a máquina en frío con colores similares del revés; secar colgado.',
                    'fr': 'Lavage en machine à froid sur l\'envers avec couleurs similaires; séchage sur cintre.',
                    'de': 'Maschinenwäsche kalt auf links mit ähnlichen Farben; hängend trocknen.',
                    'it': 'Lavare in lavatrice a freddo a rovescio con colori simili; asciugare all\'aria.',
                    'pt': 'Lavar à máquina em água fria pelo avesso com cores semelhantes; secar no varal.',
                    'ru': 'Стирать в холодной воде наизнанку с похожими цветами; сушить на вешалке.',
                    'zh': '反面朝外冷水轻柔机洗，同色系洗涤，悬挂晾干。',
                    'ja': '同系色と一緒に裏返して冷水で洗濯機洗いし、陰干ししてください。',
                    'hi': 'समान रंगों के साथ ठंडे पानी में उल्टा करके धोएं; टांग कर सुखाएं।',
                    'nl': 'Binnenstebuiten koud wassen met gelijke kleuren; hangend laten drogen.',
                },
            },
        }

        def generate_fluent_caption(color_trans, item_trans, dress_code_key, lang, sample_idx):
            """Generate rich, natural, grammatically correct captions across all 13 languages with NO English leakage."""
            d_dict = TRANSLATIONS['dress_code'].get(dress_code_key, TRANSLATIONS['dress_code']['casual'])
            d_term = d_dict.get(lang, d_dict['en'])
            style_idx = sample_idx % 4

            if lang == 'he':
                is_plural = any(w in item_trans for w in ("ג'ינס", "מכנסיים", "נעליים", "מגפיים", "סניקרס", "סנדלים"))
                if is_plural:
                    templates = [
                        f"{item_trans} בצבע {color_trans} בעיצוב נקי ואיכותי, מתאימים במיוחד ללבוש {d_term}.",
                        f"{item_trans} בגוון {color_trans} העשויים מחומרים מובחרים, המעניקים מראה מלוטש ונוח לכל שעות היום.",
                        f"{item_trans} איכותיים בצבע {color_trans} בגזרה מחמיאה, מושלמים לשילוב במגוון אאוטפיטים.",
                        f"{item_trans} אופנתיים בגוון {color_trans} המשלבים נוחות יומיומית עם סגנון {d_term} מרשים.",
                    ]
                else:
                    templates = [
                        f"{item_trans} בצבע {color_trans} בעיצוב נקי ואיכותי, מתאים במיוחד ללבוש {d_term}.",
                        f"{item_trans} בגוון {color_trans} עשוי מחומרים מובחרים, המעניק מראה מלוטש ונוח לכל שעות היום.",
                        f"{item_trans} איכותי בצבע {color_trans} בגזרה מחמיאה, מושלם לשילוב במגוון אאוטפיטים.",
                        f"{item_trans} אופנתי בגוון {color_trans} המשלב נוחות יומיומית עם סגנון {d_term} מרשים.",
                    ]
            elif lang == 'ar':
                templates = [
                    f"{item_trans} بلون {color_trans} وتصميم متقن، مثالي للإطلالات ال{d_term}.",
                    f"{item_trans} أنيق بلون {color_trans} وقصة مريحة تناسب الاستخدام اليومي الراقي.",
                    f"{item_trans} مميز بلون {color_trans} يمنحك مظهراً جذاباً وتنسيقاً سهلاً مع مختلف القطع.",
                    f"{item_trans} راقٍ بلون {color_trans} يجمع بين الراحة والأناقة لمظهر {d_term}.",
                ]
            elif lang == 'es':
                templates = [
                    f"{item_trans} de color {color_trans.lower()} con un diseño elegante, ideal para un estilo {d_term}.",
                    f"{item_trans} en tono {color_trans.lower()} confeccionado con materiales cómodos y corte favorecedor.",
                    f"{item_trans} versátil en color {color_trans.lower()}, perfecto para combinar en diversas ocasiones.",
                    f"{item_trans} clásico de color {color_trans.lower()} que combina comodidad y estilo {d_term}.",
                ]
            elif lang == 'fr':
                templates = [
                    f"{item_trans} de couleur {color_trans.lower()} au design soigné, idéal pour un style {d_term}.",
                    f"{item_trans} en teinte {color_trans.lower()} confectionné dans une coupe flatteuse et confortable.",
                    f"{item_trans} polyvalent en {color_trans.lower()}, parfait pour compléter différentes tenues du quotidien.",
                    f"{item_trans} élégant de couleur {color_trans.lower()} alliant confort moderne et allure {d_term}.",
                ]
            elif lang == 'de':
                templates = [
                    f"{item_trans} in {color_trans} mit feiner Schnittführung, ideal für einen {d_term}-Look.",
                    f"{item_trans} im Farbton {color_trans}, hergestellt aus angenehmem Material mit hohem Tragekomfort.",
                    f"Vielseitiges {item_trans} in {color_trans}, das sich mühelos zu verschiedenen Outfits kombinieren lässt.",
                    f"Klassisches {item_trans} in {color_trans}, das modernen Stil mit zeitloser Eleganz verbindet.",
                ]
            elif lang == 'it':
                templates = [
                    f"{item_trans} color {color_trans.lower()} dal design raffinato, perfetto per uno stile {d_term}.",
                    f"{item_trans} in tonalità {color_trans.lower()} caratterizzato da una vestibilità comoda e impeccabile.",
                    f"{item_trans} versatile color {color_trans.lower()}, ideale da abbinare a diversi capi del guardaroba.",
                    f"Elegante {item_trans} in {color_trans.lower()} che unisce praticità quotidiana e stile {d_term}.",
                ]
            elif lang == 'pt':
                templates = [
                    f"{item_trans} na cor {color_trans.lower()} com corte refinado, ideal para um estilo {d_term}.",
                    f"{item_trans} em tom {color_trans.lower()} confeccionado com tecido macio e caimento confortável.",
                    f"{item_trans} versátil na cor {color_trans.lower()}, perfeito para compor diversos looks modernos.",
                    f"Elegante {item_trans} em {color_trans.lower()} que combina sofisticação e conforto {d_term}.",
                ]
            elif lang == 'ru':
                templates = [
                    f"{item_trans} цвета {color_trans.lower()} с продуманным кроем, идеально подходит для стиля {d_term}.",
                    f"{item_trans} в оттенке {color_trans.lower()} из комфортного материала с отличной посадкой по фигуре.",
                    f"Универсальный предмет гардероба: {item_trans} цвета {color_trans.lower()}, легко сочетающийся с другими вещами.",
                    f"Стильный {item_trans} цвета {color_trans.lower()}, сочетающий в себе современный комфорт и практичность.",
                ]
            elif lang == 'zh':
                templates = [
                    f"{color_trans}{item_trans}，剪裁利落大方，非常适合{d_term}场合穿着。",
                    f"{color_trans}{item_trans}，面料舒适亲肤，版型挺括有型，提升日常穿搭质感。",
                    f"百搭经典的{color_trans}{item_trans}，轻松搭配多种衣橱单品，展现优雅气质。",
                    f"做工精细的{color_trans}{item_trans}，兼具日常舒适度与现代时尚美感。",
                ]
            elif lang == 'ja':
                templates = [
                    f"洗練されたデザインの{color_trans}の{item_trans}、{d_term}なシーンに最適です。",
                    f"着心地の良い素材と美しいシルエットが特徴の{color_trans}の{item_trans}。",
                    f"様々なコーディネートに着回しが効く、実用的で上品な{color_trans}の{item_trans}。",
                    f"快適さと洗練されたスタイルを兼ね備えた、クラシックな{color_trans}の{item_trans}。",
                ]
            elif lang == 'hi':
                templates = [
                    f"{color_trans} रंग का {item_trans} सुरुचिपूर्ण डिज़ाइन के साथ, {d_term} पहनावे के लिए उत्तम।",
                    f"आरामदायक फ़ैब्रिक और बेहतरीन फ़िटिंग वाला {color_trans} {item_trans}, पूरे दिन के आराम के लिए।",
                    f"बहुमुखी {color_trans} {item_trans}, जिसे विभिन्न परिधानों के साथ आसानी से स्टाइल किया जा सकता है।",
                    f"क्लासिक {color_trans} {item_trans} जो आधुनिक आराम और आकर्षक स्टाइल का आदर्श मेल है।",
                ]
            elif lang == 'nl':
                templates = [
                    f"{item_trans} in de kleur {color_trans.lower()} met een verzorgde pasvorm, ideaal voor een {d_term} stijl.",
                    f"{item_trans} in {color_trans.lower()} gemaakt van comfortabel materiaal voor optimaal draagcomfort.",
                    f"Veelzijdige {item_trans} in {color_trans.lower()}, moeiteloos te combineren met verschillende garderobe-items.",
                    f"Stijlvolle {item_trans} in {color_trans.lower()} die dagelijks gemak combineert met een tijdloze uitstraling.",
                ]
            else:
                templates = [
                    f"A refined {color_trans.lower()} {item_trans.lower()} tailored for effortless {d_term} styling.",
                    f"Crafted in {color_trans.lower()} with a comfortable texture and flattering, versatile silhouette.",
                    f"A versatile {color_trans.lower()} {item_trans.lower()} that coordinates seamlessly across multiple wardrobe looks.",
                    f"Timeless {color_trans.lower()} {item_trans.lower()} combining all-day comfort with contemporary style.",
                ]
            res = templates[style_idx]
            if not res.endswith('.'):
                res += '.'
            return res

        def get_realistic_fabrics(clean_article, cat, sub_cat, desc_name):
            """Assign realistic fabric compositions summing strictly to 100%."""
            low = f"{clean_article} {cat} {sub_cat} {desc_name}".lower()
            if "denim" in low or "jean" in low:
                return [{"name": "cotton", "pct": 98}, {"name": "elastane", "pct": 2}]
            elif "leather" in low or "biker" in low or "moto" in low:
                return [{"name": "leather", "pct": 100}]
            elif "suede" in low:
                return [{"name": "suede", "pct": 100}]
            elif "wool" in low or "blazer" in low or "tweed" in low or "suit" in low:
                return [{"name": "wool", "pct": 70}, {"name": "polyester", "pct": 30}]
            elif "silk" in low or "satin" in low:
                return [{"name": "silk", "pct": 100}]
            elif "linen" in low:
                return [{"name": "linen", "pct": 100}]
            elif "velvet" in low or "corduroy" in low:
                return [{"name": "cotton", "pct": 80}, {"name": "polyester", "pct": 20}]
            elif "chiffon" in low or "lace" in low:
                return [{"name": "polyester", "pct": 60}, {"name": "viscose", "pct": 40}]
            elif "sweater" in low or "knit" in low or "cardigan" in low:
                return [{"name": "wool", "pct": 80}, {"name": "polyamide", "pct": 20}]
            elif "track" in low or "sport" in low or "running" in low or "jogger" in low or "legging" in low:
                return [{"name": "polyester", "pct": 85}, {"name": "elastane", "pct": 15}]
            elif "puffer" in low or "down" in low or "parka" in low or "windbreaker" in low:
                return [{"name": "nylon", "pct": 100}]
            elif "coat" in low or "jacket" in low:
                return [{"name": "wool", "pct": 60}, {"name": "polyester", "pct": 40}]
            elif "chino" in low or "trouser" in low or "pant" in low:
                return [{"name": "cotton", "pct": 97}, {"name": "elastane", "pct": 3}]
            elif cat == "Footwear":
                if any(w in low for w in ("sneaker", "canvas", "slip-on")):
                    return [{"name": "canvas", "pct": 70}, {"name": "rubber", "pct": 30}]
                return [{"name": "leather", "pct": 75}, {"name": "rubber", "pct": 25}]
            elif cat == "Accessories":
                if "bag" in low or "clutch" in low or "purse" in low or "tote" in low:
                    return [{"name": "leather", "pct": 100}]
                elif "belt" in low:
                    return [{"name": "leather", "pct": 100}]
                elif "sunglass" in low or "watch" in low:
                    return [{"name": "metal", "pct": 60}, {"name": "acetate", "pct": 40}]
                return [{"name": "cotton", "pct": 100}]
            elif "dress" in low or "skirt" in low or "blouse" in low:
                return [{"name": "viscose", "pct": 65}, {"name": "polyester", "pct": 35}]
            else:
                return [{"name": "cotton", "pct": 100}]

        def localize_title(color_key, item_key, lang):
            c_dict = TRANSLATIONS['colors'].get(color_key.lower(), {})
            c_trans = c_dict.get(lang, c_dict.get('en', color_key))
            i_dict = TRANSLATIONS['items'].get(item_key, {})
            i_trans = i_dict.get(lang, i_dict.get('en', item_key))
            if lang in ('he', 'ar'):
                return f"{i_trans} {c_trans}"
            elif lang in ('zh', 'ja'):
                return f"{c_trans}{i_trans}"
            else:
                return f"{c_trans} {i_trans}"

        def get_synthetic_gender(clean_article, cat, sub_cat, desc_name, row_gender):
            low = f"{clean_article} {cat} {sub_cat} {desc_name}".lower()
            if any(k in low for k in (
                "heels", "pumps", "stiletto", "skirt", "dress", "blouse", "handbag",
                "purse", "clutch", "tote bag", "flats", "strappy sandals", "gown", "sundress"
            )) or row_gender == 'Women':
                return "women"
            elif any(k in low for k in ("tuxedo", "boxer", "tie", "cufflink")) or row_gender == 'Men':
                if any(k in low for k in ("heels", "skirt", "dress", "blouse", "handbag", "purse")):
                    return "women"
                return "men"
            else:
                return "unisex"

        # ===========================================================================
        # PART A: ashraq/fashion-product-images-small (Isolated Studio Cutouts)
        # ===========================================================================
        print('Loading ashraq/fashion-product-images-small...')
        ds_ashraq = load_dataset('ashraq/fashion-product-images-small', split='train')
        print(f'Loaded {len(ds_ashraq)} studio items.')

        CATEGORY_RULES = {
            'top': {'articles': ['Tshirts', 'Shirts', 'Tops', 'Sweaters', 'Sweatshirts', 'Kurtas', 'Tunics', 'Blouses'], 'cat': 'Top'},
            'bottom': {'articles': ['Jeans', 'Trousers', 'Shorts', 'Skirts', 'Track Pants', 'Leggings', 'Capris'], 'cat': 'Bottom'},
            'footwear': {'articles': ['Casual Shoes', 'Sports Shoes', 'Heels', 'Flats', 'Sandals', 'Boots', 'Flip Flops'], 'cat': 'Footwear'},
            'accessory': {'articles': ['Sunglasses', 'Handbags', 'Backpacks', 'Clutches', 'Belts', 'Caps', 'Watches', 'Scarves', 'Wallets'], 'cat': 'Accessories'},
            'outerwear_dress': {'articles': ['Dresses', 'Jackets', 'Blazers', 'Coats', 'Jumpsuit', 'Shrug', 'Rompers'], 'cat': 'Outerwear'},
        }
        ashraq_counts = {k: 0 for k in CATEGORY_RULES}
        ASHRAQ_LIMIT = 400  # 400 * 5 = 2,000 samples

        for idx, row in enumerate(tqdm(ds_ashraq, desc='Processing Studio Cutouts (13 Languages)')) :
            article = row.get('articleType', '')
            master = row.get('masterCategory', '')
            desc_name = str(row.get('productDisplayName', '')).lower()
            matched = None
            for b_name, b_spec in CATEGORY_RULES.items():
                if article in b_spec['articles'] or (b_name == 'accessory' and master == 'Accessories'):
                    if ashraq_counts[b_name] < ASHRAQ_LIMIT:
                        matched = b_name
                        break
            if matched:
                ashraq_counts[matched] += 1
                target_lang = LANGUAGES[len(dataset_samples) % len(LANGUAGES)]
                color = row.get('baseColour') or 'Black'
                color_key = color.lower() if color.lower() in TRANSLATIONS['colors'] else 'black'
                gender_raw = row.get('gender') or 'Unisex'
                usage = row.get('usage') or 'Casual'
                dress_code = 'smart-casual' if usage in ('Smart Casual', 'Ethnic') else ('formal' if usage in ('Formal', 'Party') else ('athletic' if usage == 'Sports' else 'casual'))
                
                # High-precision subcategory and item_type mapping
                if article == 'Heels':
                    clean_article = 'High Heel Pumps' if 'pump' in desc_name else ('Stiletto Heels' if 'stiletto' in desc_name else 'Heeled Pumps')
                    sub_category = 'Heels'
                elif article == 'Flats':
                    clean_article = 'Ballet Flats'
                    sub_category = 'Flats'
                elif article == 'Casual Shoes':
                    clean_article = 'Leather Loafers' if 'loafer' in desc_name else ('Oxford Dress Shoes' if 'oxford' in desc_name else 'Casual Slip-Ons')
                    sub_category = 'Loafers'
                elif article == 'Sports Shoes':
                    clean_article = 'Running Sneakers'
                    sub_category = 'Sneakers'
                elif article == 'Boots':
                    clean_article = 'Ankle Boots' if 'ankle' in desc_name else 'Leather Boots'
                    sub_category = 'Boots'
                elif article == 'Sandals':
                    clean_article = 'Strappy Sandals'
                    sub_category = 'Sandals'
                elif article == 'Skirts':
                    clean_article = 'A-Line Skirt' if 'a-line' in desc_name else ('Maxi Skirt' if 'maxi' in desc_name else 'Pleated Skirt')
                    sub_category = 'Skirts'
                elif article == 'Trousers':
                    clean_article = 'Tailored Chinos' if 'chino' in desc_name else 'Tailored Trousers'
                    sub_category = 'Pants'
                elif article == 'Jeans':
                    clean_article = 'Slim-Fit Denim Jeans'
                    sub_category = 'Jeans'
                elif article == 'Tshirts':
                    clean_article = 'Crewneck Cotton T-Shirt'
                    sub_category = 'T-Shirts'
                elif article == 'Shirts':
                    clean_article = 'Button-Down Shirt'
                    sub_category = 'Shirts'
                elif article == 'Dresses':
                    clean_article = 'Maxi Dress' if 'maxi' in desc_name else 'A-Line Casual Dress'
                    sub_category = 'Dresses'
                elif article in ('Handbags', 'Backpacks', 'Clutches'):
                    clean_article = 'Leather Handbag' if 'leather' in desc_name or article == 'Handbags' else ('Canvas Tote Bag' if 'tote' in desc_name else 'Crossbody Bag')
                    sub_category = 'Handbags'
                elif article in ('Jackets', 'Blazers', 'Coats'):
                    clean_article = 'Tailored Blazer' if 'blazer' in desc_name else 'Structured Jacket'
                    sub_category = 'Jackets'
                elif 'blouse' in desc_name:
                    clean_article = 'Silk Blouse'
                    sub_category = 'Blouses'
                else:
                    clean_article = 'Crewneck Cotton T-Shirt' if matched == 'top' else ('Tailored Chinos' if matched == 'bottom' else 'Casual Slip-Ons')
                    sub_category = CATEGORY_RULES[matched]['cat']

                cat_name = CATEGORY_RULES[matched]['cat']
                gender = get_synthetic_gender(clean_article, cat_name, sub_category, desc_name, gender_raw)
                localized_title = localize_title(color_key, clean_article, target_lang)
                care_type = 'footwear' if matched == 'footwear' else 'apparel'
                care_text = TRANSLATIONS['care'][care_type].get(target_lang, TRANSLATIONS['care'][care_type]['en'])
                c_name = TRANSLATIONS['colors'].get(color_key, {}).get(target_lang, color)
                i_name = TRANSLATIONS['items'].get(clean_article, {}).get(target_lang, clean_article)
                caption_text = generate_fluent_caption(c_name, i_name, dress_code, target_lang, len(dataset_samples))
                fabric_list = get_realistic_fabrics(clean_article, cat_name, sub_category, desc_name)

                img_path = dataset_dir / f'ashraq_{idx}.jpg'
                row['image'].save(img_path, 'JPEG')

                gt_json = {
                    'is_clothing': True,
                    'name': localized_title,
                    'title': localized_title,
                    'category': cat_name,
                    'sub_category': sub_category,
                    'item_type': clean_article,
                    'caption': caption_text,
                    'gender': gender,
                    'model_gender': None,
                    'dress_code': dress_code,
                    'season': ['spring', 'summer', 'fall', 'winter'],
                    'colors': [{'name': color_key, 'pct': 100}],
                    'fabric_materials': fabric_list,
                    'pattern': 'solid',
                    'cleaning_tips': care_text,
                    'care_instructions': care_text,
                    'tags': [c_name.lower(), i_name.lower(), dress_code],
                }
                dataset_samples.append({
                    'image_path': str(img_path),
                    'system': PROMPT_GARMENT_VISION,
                    'user': get_user_prompt(target_lang),
                    'assistant': json.dumps(gt_json, ensure_ascii=False),
                })
                if all(c >= ASHRAQ_LIMIT for c in ashraq_counts.values()):
                    break

        # ===========================================================================
        # PART B: Marqo/deepfashion-multimodal (Textures, Fabrics & Complex Patterns)
        # ===========================================================================
        print('Streaming Marqo/deepfashion-multimodal for complex textures and patterns (2,000 samples)...')
        try:
            ds_marqo = load_dataset('Marqo/deepfashion-multimodal', split='train', streaming=True)
            marqo_count = 0
            MARQO_LIMIT = 2000
            for idx, row in enumerate(tqdm(ds_marqo, total=MARQO_LIMIT, desc='Processing DeepFashion Multimodal (13 Languages)')):
                if marqo_count >= MARQO_LIMIT:
                    break
                img = row.get('image')
                desc = row.get('description') or row.get('text') or ''
                if not img or not desc:
                    continue
                
                desc_low = str(desc).lower()
                target_lang = LANGUAGES[len(dataset_samples) % len(LANGUAGES)]
                
                pattern = 'solid'
                for pat_cand in ('camouflage', 'striped', 'plaid', 'floral', 'geometric', 'houndstooth', 'printed'):
                    if pat_cand in desc_low:
                        pattern = pat_cand
                        break
                
                color_key = 'navy' if 'blue' in desc_low else ('olive' if 'green' in desc_low else ('brown' if 'brown' in desc_low else ('white' if 'white' in desc_low else 'black')))
                
                if 'skirt' in desc_low:
                    cat = 'Bottom'
                    sub_cat = 'Skirts'
                    clean_article = 'A-Line Skirt' if 'a-line' in desc_low else ('Maxi Skirt' if 'maxi' in desc_low else 'Pleated Skirt')
                elif 'heel' in desc_low or 'pump' in desc_low:
                    cat = 'Footwear'
                    sub_cat = 'Heels'
                    clean_article = 'High Heel Pumps'
                elif 'boot' in desc_low:
                    cat = 'Footwear'
                    sub_cat = 'Boots'
                    clean_article = 'Ankle Boots'
                elif 'sneaker' in desc_low or 'trainer' in desc_low:
                    cat = 'Footwear'
                    sub_cat = 'Sneakers'
                    clean_article = 'Running Sneakers'
                elif 'dress' in desc_low:
                    cat = 'Outerwear'
                    sub_cat = 'Dresses'
                    clean_article = 'Maxi Dress' if 'maxi' in desc_low else 'A-Line Casual Dress'
                elif 'blazer' in desc_low or 'jacket' in desc_low or 'coat' in desc_low:
                    cat = 'Outerwear'
                    sub_cat = 'Jackets'
                    clean_article = 'Tailored Blazer' if 'blazer' in desc_low else 'Structured Jacket'
                elif 'pant' in desc_low or 'trouser' in desc_low or 'jean' in desc_low:
                    cat = 'Bottom'
                    sub_cat = 'Jeans' if 'jean' in desc_low else 'Pants'
                    clean_article = 'Slim-Fit Denim Jeans' if 'jean' in desc_low else ('Cargo Pants' if 'cargo' in desc_low else 'Tailored Chinos')
                else:
                    cat = 'Top'
                    sub_cat = 'Tops'
                    clean_article = 'Silk Blouse' if 'silk' in desc_low or 'blouse' in desc_low else 'Cotton Shirt'

                img_path = dataset_dir / f'marqo_{idx}.jpg'
                if hasattr(img, 'save'):
                    img.save(img_path, 'JPEG')
                else:
                    continue
                
                marqo_count += 1
                dress_code = 'casual' if 'casual' in desc_low else 'smart-casual'
                gender = 'women' if 'woman' in desc_low or 'female' in desc_low or cat == 'Outerwear' and sub_cat == 'Dresses' or sub_cat in ('Skirts', 'Heels', 'Blouses') else ('men' if 'man' in desc_low or 'male' in desc_low else 'unisex')
                model_gender = 'women' if ('woman' in desc_low or 'female' in desc_low) else ('men' if ('man' in desc_low or 'male' in desc_low) else None)
                localized_title = localize_title(color_key, clean_article, target_lang)
                care_type = 'footwear' if cat == 'Footwear' else 'apparel'
                care_text = TRANSLATIONS['care'][care_type].get(target_lang, TRANSLATIONS['care'][care_type]['en'])
                c_name = TRANSLATIONS['colors'].get(color_key, {}).get(target_lang, color_key)
                i_name = TRANSLATIONS['items'].get(clean_article, {}).get(target_lang, clean_article)
                caption_text = generate_fluent_caption(c_name, i_name, dress_code, target_lang, len(dataset_samples))
                fabric_list = get_realistic_fabrics(clean_article, cat, sub_cat, desc_low)
                
                gt_json = {
                    'is_clothing': True,
                    'name': localized_title,
                    'title': localized_title,
                    'category': cat,
                    'sub_category': sub_cat,
                    'item_type': clean_article,
                    'caption': caption_text,
                    'gender': gender,
                    'model_gender': model_gender,
                    'dress_code': dress_code,
                    'season': ['spring', 'summer', 'fall', 'winter'],
                    'colors': [{'name': color_key, 'pct': 100}],
                    'fabric_materials': fabric_list,
                    'pattern': pattern,
                    'cleaning_tips': care_text,
                    'care_instructions': care_text,
                    'tags': [c_name.lower(), i_name.lower(), pattern],
                }
                dataset_samples.append({
                    'image_path': str(img_path),
                    'system': PROMPT_GARMENT_VISION,
                    'user': get_user_prompt(target_lang),
                    'assistant': json.dumps(gt_json, ensure_ascii=False),
                })
        except Exception as e:
            print(f'Notice on Marqo/deepfashion-multimodal: {e}')

        # ===========================================================================
        # PART C: ArtmeScienceLab/Garments2Look (Outfit Coordination & Full Ensembles)
        # ===========================================================================
        print('Streaming ArtmeScienceLab/Garments2Look for complete head-to-toe outfits (2,000 samples)...')
        try:
            ds_g2l = load_dataset('ArtmeScienceLab/Garments2Look', split='train', streaming=True)
            g2l_count = 0
            G2L_LIMIT = 2000
            for idx, row in enumerate(tqdm(ds_g2l, total=G2L_LIMIT, desc='Processing Garments2Look Outfits (13 Languages)')):
                if g2l_count >= G2L_LIMIT:
                    break
                look_img = row.get('look_image') or row.get('image')
                if not look_img:
                    continue
                
                img_path = dataset_dir / f'g2l_{idx}.jpg'
                if hasattr(look_img, 'save'):
                    look_img.save(img_path, 'JPEG')
                else:
                    continue
                
                g2l_count += 1
                target_lang = LANGUAGES[len(dataset_samples) % len(LANGUAGES)]
                
                # Localized scheduled outfit descriptions
                vibe_map = {
                    'en': 'Smart Casual Everyday Chic',
                    'he': 'סמארט קז\'ואל שיקי ליומיום',
                    'ar': 'سمارت كاجوال أنيق للاستخدام اليومي',
                    'es': 'Smart Casual Chic diario',
                    'fr': 'Smart Casual Chic du quotidien',
                    'de': 'Smart Casual Alltags-Chic',
                    'it': 'Smart Casual Chic per tutti i giorni',
                    'pt': 'Smart Casual Chic para o dia a dia',
                    'ru': 'Смарт-кэжуал повседневный шик',
                    'zh': '日常优雅轻商务风',
                    'ja': 'スマートカジュアル・デイリーシック',
                    'hi': 'स्मार्ट कैज़ुअल दैनिक पहनावा',
                    'nl': 'Smart casual alledaagse chic',
                }
                weather_map = {
                    'en': 'Balanced breathable layers comfortable for 22°C mild forecast.',
                    'he': 'שכבות נושמות ומאוזנות המתאימות לתחזית נעימה של 22 מעלות.',
                    'ar': 'طبقات مريحة ومسامية تناسب طقساً معتدلاً عند 22 درجة مئوية.',
                    'es': 'Capas transpirables y equilibradas cómodas para 22°C.',
                    'fr': 'Superposition équilibrée et respirante confortable pour 22°C.',
                    'de': 'Atmungsaktive Lagen, optimal für milde 22°C.',
                    'it': 'Strati traspiranti ed equilibrati per clima mite a 22°C.',
                    'pt': 'Camadas respiráveis e confortáveis para clima ameno de 22°C.',
                    'ru': 'Сбалансированные дышащие слои для комфортных 22°C.',
                    'zh': '透气多层次搭配，适合22度温和天气。',
                    'ja': '気温22度の過ごしやすい気候に適した通気性レイヤード。',
                    'hi': '22 डिग्री सेल्सियस के अनुकूल संतुलित हवादार परतें।',
                    'nl': 'Ademende kledinglagen geschikt voor een milde 22°C.',
                }
                top_name = TRANSLATIONS['items']['Cotton Shirt'].get(target_lang, 'Cotton Shirt')
                bot_name = TRANSLATIONS['items']['Tailored Chinos'].get(target_lang, 'Tailored Chinos')
                shoe_name = TRANSLATIONS['items']['Oxford Dress Shoes'].get(target_lang, 'Oxford Dress Shoes')
                title_trans = f'Look {g2l_count}' if target_lang not in ('he','ar') else f'מראה {g2l_count}'
                
                proposal = {
                    'proposals': [{
                        'date': '2026-10-04',
                        'title': f'{title_trans} ({vibe_map.get(target_lang, vibe_map["en"])})',
                        'items': [
                            {'role': 'top', 'closet_item_id': f'item_top_{idx}', 'name': top_name},
                            {'role': 'bottom', 'closet_item_id': f'item_bot_{idx}', 'name': bot_name},
                            {'role': 'shoes', 'closet_item_id': f'item_shoe_{idx}', 'name': shoe_name},
                        ],
                        'vibe': vibe_map.get(target_lang, vibe_map['en']),
                        'weather_rationale': weather_map.get(target_lang, weather_map['en']),
                    }]
                }
                dataset_samples.append({
                    'image_path': str(img_path),
                    'system': PROMPT_SCHEDULED_OUTFIT,
                    'user': get_user_prompt(target_lang),
                    'assistant': json.dumps(proposal, ensure_ascii=False),
                })
        except Exception as e:
            print(f'Notice on ArtmeScienceLab/Garments2Look: {e}')

        print(f'✅ Tri-Dataset ETL Complete! Total curated multilingual samples: {len(dataset_samples)}')

        # Export final JSONL dataset
        with open('curated_dataset.jsonl', 'w', encoding='utf-8') as f:
            for s in dataset_samples:
                f.write(json.dumps(s, ensure_ascii=False) + '\n')
        print(f'Dataset exported to curated_dataset.jsonl ({Path("curated_dataset.jsonl").stat().st_size / 1024:.1f} KB)')


## 🚀 Step 4: Fine-Tuning Gemma-4 E4B with QLoRA & Prompt Loss Masking
Fine-tune `google/gemma-4-e4b-it` using 4-bit BitsAndBytes quantization and PEFT LoRA.
- Formats with native Gemma-4 chat template (`<start_of_turn>user` / `<start_of_turn>model`).
- Masks prompt loss (`labels[:prompt_len] = -100`) so the model only learns to output accurate JSON.
- Preserves native multimodal visual embeddings without distortion.


In [ ]:
# Ensure bitsandbytes 4-bit CUDA quantization backend is up-to-date
!pip install -q -U 'bitsandbytes>=0.46.1'

# Force disable TensorFlow in Transformers runtime cache
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
import transformers.utils.import_utils
transformers.utils.import_utils._tf_available = False
transformers.utils.is_tf_available = lambda: False
transformers.utils.import_utils.is_tf_available = lambda: False

import sys
for mod in list(sys.modules.keys()):
    if mod.startswith('tensorflow') or mod.startswith('tensorboard'):
        del sys.modules[mod]

# Defensive verification and memory purge for Pillow typing compatibility
try:
    import PIL._typing
    if not hasattr(PIL._typing, '_Ink'):
        for mod in list(sys.modules.keys()):
            if mod.startswith('PIL'):
                del sys.modules[mod]
except Exception:
    for mod in list(sys.modules.keys()):
        if mod.startswith('PIL'):
            del sys.modules[mod]

import getpass
import json
from PIL import Image
import torch
from transformers import AutoProcessor, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Patch transformers tokenizer to handle list-type extra_special_tokens in Gemma-4 config (non-recursive)
import transformers.tokenization_utils_base
def _safe_set_model_specific_special_tokens(self, special_tokens=None):
    if special_tokens is not None:
        if isinstance(special_tokens, list):
            special_tokens = {str(tok): tok for tok in special_tokens}
        elif not isinstance(special_tokens, dict):
            return
        attrs = getattr(self, 'SPECIAL_TOKENS_ATTRIBUTES', [])
        self.SPECIAL_TOKENS_ATTRIBUTES = list(set(attrs + list(special_tokens.keys())))
        for key, value in special_tokens.items():
            try:
                setattr(self, str(key), value)
            except Exception:
                pass
transformers.tokenization_utils_base.PreTrainedTokenizerBase._set_model_specific_special_tokens = _safe_set_model_specific_special_tokens
MODEL_ID = 'google/gemma-4-e4b-it'

# Automatically retrieve token from Colab Secrets ('HF_Gemma4' or 'HF_TOKEN') without prompt
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_Gemma4') or userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_Gemma4') or os.environ.get('HF_TOKEN')

# Verify GPU is available
if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU detected! In Colab, go to menu: Runtime -> Change runtime type -> Hardware accelerator: T4 GPU')

# Check if bitsandbytes is available for 4-bit quantization, otherwise fallback to FP16
use_4bit = False
try:
    import bitsandbytes as bnb
    from transformers.utils import is_bitsandbytes_available
    if is_bitsandbytes_available():
        use_4bit = True
except Exception:
    pass

# Robust initialization of Gemma-4 multimodal processor and model architecture
try:
    from transformers import Gemma4Processor, Gemma4Config, Gemma4ForConditionalGeneration
except (ImportError, ModuleNotFoundError):
    print('⚠️ Gemma-4 architecture missing from active transformers. Updating dependencies and installing bleeding-edge transformers...')
    !pip install -q -U 'huggingface_hub==1.33.0' 'tokenizers==0.23.2'
    !pip install -q -U --force-reinstall git+https://github.com/huggingface/transformers.git
    print('🔄 Kernel reload required to activate Gemma-4 architecture. Reloading...')
    import os
    os.kill(os.getpid(), 9)

from transformers import Gemma4Processor, Gemma4Config, Gemma4ForConditionalGeneration
from transformers.models.auto.processing_auto import PROCESSOR_MAPPING
PROCESSOR_MAPPING[Gemma4Config] = Gemma4Processor
processor = Gemma4Processor.from_pretrained(MODEL_ID, token=HF_TOKEN)
ModelClass = Gemma4ForConditionalGeneration
print('✅ Successfully loaded native Gemma4Processor and Gemma4ForConditionalGeneration!')

if use_4bit:
    print('✅ Loading model in 4-bit NF4 (bitsandbytes enabled)...')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = ModelClass.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
    )
else:
    print('ℹ️ bitsandbytes not available — loading model directly in FP16 with gradient checkpointing...')
    model = ModelClass.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map='auto',
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

# LoRA Configuration targeting language projections while freezing vision embeddings
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    exclude_modules=r'.*(vision_tower|audio_tower|embed_vision|embed_audio).*',
)

# Remove incompatible legacy torchao from Colab environment if present
!pip uninstall -y torchao > /dev/null 2>&1
import sys
for mod in list(sys.modules.keys()):
    if 'torchao' in mod:
        del sys.modules[mod]

model = get_peft_model(model, lora_config)
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model.print_trainable_parameters()

# PyTorch Multimodal Dataset with Native Chat Template and Prompt Loss Masking
class FashionMultimodalDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_file, processor, max_samples=None):
        self.items = []
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                self.items.append(json.loads(line))
        if max_samples:
            self.items = self.items[:max_samples]
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img = Image.open(item['image_path']).convert('RGB')
        
        # Format messages using standard multimodal conversation format
        messages_prompt_only = [
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': f"{item['system']}\n\n{item['user']}"}
                ]
            }
        ]
        messages_full = [
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': f"{item['system']}\n\n{item['user']}"}
                ]
            },
            {
                'role': 'assistant',
                'content': [
                    {'type': 'text', 'text': item['assistant']}
                ]
            }
        ]
        
        try:
            prompt_text = self.processor.apply_chat_template(messages_prompt_only, tokenize=False, add_generation_prompt=True)
            full_text = self.processor.apply_chat_template(messages_full, tokenize=False)
        except Exception:
            # Fallback format if custom jinja template has different token names
            image_token = getattr(self.processor, 'image_token', '<|image|>') or '<|image|>'
            prompt_text = f"<start_of_turn>user\n{image_token}\n{item['system']}\n\n{item['user']}<end_of_turn>\n<start_of_turn>model\n"
            full_text = f"{prompt_text}{item['assistant']}<end_of_turn>\n"
        
        # Compute prompt length boundary for loss masking
        prompt_inputs = self.processor(text=prompt_text, images=img, return_tensors='pt')
        prompt_len = prompt_inputs['input_ids'].shape[1]
        
        # Tokenize complete user + assistant sequence
        inputs = self.processor(text=full_text, images=img, return_tensors='pt')
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        labels = inputs['input_ids'].clone()
        
        # Mask prompt tokens (-100) so gradients only update on the assistant JSON response
        mask_boundary = min(prompt_len, len(labels))
        labels[:mask_boundary] = -100
        inputs['labels'] = labels
        return inputs

train_dataset = FashionMultimodalDataset('curated_dataset.jsonl', processor, max_samples=6000)

# Custom collator for multimodal batches with batch_size=1
def collate_fn(batch):
    elem = batch[0]
    return {k: torch.stack([d[k] for d in batch]) if isinstance(elem[k], torch.Tensor) else [d[k] for d in batch] for k in elem}

# Training Arguments (6,000 samples, 1,200 steps with BF16/FP16)
training_args = TrainingArguments(
    output_dir='output_eyes_lora',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=25,
    max_steps=1200,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    save_strategy='steps',
    save_steps=300,
    save_total_limit=2,
    warmup_steps=60,
    dataloader_num_workers=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    args=training_args,
)

print('Starting QLoRA Multilingual Fine-Tuning...')
trainer.train()
print('✅ Fine-tuning completed!')
model.save_pretrained('output_eyes_lora/adapter')
processor.save_pretrained('output_eyes_lora/adapter')


## ⚙️ Step 5: Merge Adapter & Quantize to GGUF
Merge fine-tuned LoRA weights into the base Gemma-4 model, convert to intermediate F16 GGUF, and quantize to `Q3_K_M` (production target) and `Q4_K_M` using `llama.cpp`.


In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
import transformers.utils.import_utils
transformers.utils.import_utils._tf_available = False
transformers.utils.is_tf_available = lambda: False
transformers.utils.import_utils.is_tf_available = lambda: False
import sys
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
MODEL_ID = 'google/gemma-4-e4b-it'
if 'HF_TOKEN' not in globals() or not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_Gemma4') or userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_Gemma4') or os.environ.get('HF_TOKEN')

print('Freeing training VRAM before merge...')
try:
    del trainer, model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

print('Reloading base model in BF16 on CPU for merge...')
try:
    from transformers import Gemma4ForConditionalGeneration as BaseClass
except Exception:
    try:
        from transformers.models.gemma4.modeling_gemma4 import Gemma4ForConditionalGeneration as BaseClass
    except Exception:
        from transformers import AutoModelForImageTextToText as BaseClass

base_model = BaseClass.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Locate the trained LoRA adapter directory
adapter_path = Path('output_eyes_lora/adapter')
if not (adapter_path / 'adapter_config.json').exists():
    if (Path('output_eyes_lora') / 'adapter_config.json').exists():
        adapter_path = Path('output_eyes_lora')
    else:
        checkpoints = sorted(Path('output_eyes_lora').glob('checkpoint-*'))
        if checkpoints and (checkpoints[-1] / 'adapter_config.json').exists():
            adapter_path = checkpoints[-1]
            print(f'Using latest checkpoint: {adapter_path}')
        else:
            raise FileNotFoundError(
                '❌ Can not find adapter_config.json in output_eyes_lora!\n' +
                'Please run Step 4 (Fine-Tuning cell with trainer.train()) to completion first before running Step 5.'
            )

print(f'Loading adapter from {adapter_path}...')
merged_model = PeftModel.from_pretrained(base_model, str(adapter_path))
merged_model = merged_model.merge_and_unload()

merged_dir = Path('output_merged_hf')
merged_dir.mkdir(exist_ok=True)
merged_model.save_pretrained(str(merged_dir))
processor.save_pretrained(str(merged_dir))
print(f'✅ Merged model saved to {merged_dir}')

# Step 5b: Ensure llama.cpp is cloned and conversion dependencies installed
if not Path('llama.cpp').exists():
    print('Cloning llama.cpp repository...')
    !git clone https://github.com/ggerganov/llama.cpp.git
    !pip install -q -U gguf sentencepiece protobuf

# Step 5c: Convert to F16 GGUF
f16_path = Path('gemma-4-E4B-it-f16.gguf')
if not f16_path.exists():
    print('Converting merged HF model to F16 GGUF...')
    !python llama.cpp/convert_hf_to_gguf.py output_merged_hf --outfile gemma-4-E4B-it-f16.gguf --outtype f16

if not f16_path.exists():
    raise FileNotFoundError('❌ Failed to create gemma-4-E4B-it-f16.gguf via convert_hf_to_gguf.py. Check the output above.')
print(f'✅ F16 GGUF created successfully ({f16_path.stat().st_size / (1024**3):.2f} GB)')

# Step 5c: Locate or build llama-quantize
quantize_bin = None
for p in Path('llama.cpp').rglob('llama-quantize'):
    if p.is_file() and os.access(p, os.X_OK):
        quantize_bin = str(p.resolve())
        break

if not quantize_bin:
    print('llama-quantize not found. Compiling via cmake...')
    !cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF
    !cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize
    for p in Path('llama.cpp').rglob('llama-quantize'):
        if p.is_file() and os.access(p, os.X_OK):
            quantize_bin = str(p.resolve())
            break

if not quantize_bin:
    raise FileNotFoundError('❌ Failed to locate or compile llama-quantize!')
print(f'Using quantize binary: {quantize_bin}')

# Step 5d: Quantize to Q3_K_M (production target ~2.85 GB) and Q4_K_M
print('Quantizing to Q3_K_M (production VPS target)...')
!{quantize_bin} gemma-4-E4B-it-f16.gguf gemma-4-E4B-it-Q3_K_M.gguf Q3_K_M

print('Quantizing to Q4_K_M...')
!{quantize_bin} gemma-4-E4B-it-f16.gguf gemma-4-E4B-it-Q4_K_M.gguf Q4_K_M

!ls -lh gemma-4-E4B-it*.gguf
print('✅ GGUF quantization completed successfully!')


## 🧪 Step 6: Test & Benchmark Attribute Parsing across Languages
Validate the quantized GGUF model with DressApp's canonical JSON schema to ensure strict compliance and non-generic descriptions.


In [ ]:
import os
import json
from pathlib import Path

print('Running smoke test verification...')
# Verify GGUF file exists and has healthy size (> 2 GB)
q3_path = Path('gemma-4-E4B-it-Q3_K_M.gguf')
assert q3_path.exists(), 'Q3_K_M GGUF file was not generated!'
size_gb = q3_path.stat().st_size / (1024**3)
print(f'Q3_K_M GGUF Size: {size_gb:.2f} GB')
assert size_gb > 2.0, f'GGUF file size {size_gb:.2f} GB is unexpectedly small!'

print('✅ GGUF file passed structural and size benchmarks!')


## 🚀 Step 7: Push Quantized Weights to Hetzner VPS & Reload
Deploy the fine-tuned `gemma-4-E4B-it-Q3_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` on the VPS.
If your Colab instance's SSH key is authorized, this step uploads directly to the VPS and reloads `dressapp-eyes` in ~20 seconds.


In [ ]:
import os
from pathlib import Path

gguf_file = 'gemma-4-E4B-it-Q3_K_M.gguf'
assert Path(gguf_file).exists(), f'❌ {gguf_file} not found!'

# 1. Ensure Colab SSH key directory exists
!mkdir -p /root/.ssh
if not os.path.exists('/root/.ssh/id_ed25519'):
    !ssh-keygen -t ed25519 -N '' -f /root/.ssh/id_ed25519

# 2. Test SSH connection to VPS
print('Testing SSH connection to Hetzner VPS (178.105.144.142)...')
res = !ssh -o BatchMode=yes -o ConnectTimeout=5 -o StrictHostKeyChecking=no root@178.105.144.142 'echo connected' 2>/dev/null

if res and 'connected' in res[0]:
    print('✅ SSH Key authorized! Uploading directly to VPS at datacenter speeds...')
    !scp -o StrictHostKeyChecking=no {gguf_file} root@178.105.144.142:/srv/AI-Stylist/eyes_gguf/
    print('🔄 Restarting dressapp-eyes container...')
    !ssh -o StrictHostKeyChecking=no root@178.105.144.142 'cd /srv/AI-Stylist/deploy && docker compose restart eyes && sleep 8 && docker exec dressapp-eyes curl -s http://127.0.0.1:7860/healthz'
    print('🎉 Model successfully deployed and healthy!')
else:
    print('ℹ️ Colab SSH Key is not yet authorized on the VPS.')
    print('To enable 1-click datacenter upload, copy and paste this key into your chat:')
    !cat /root/.ssh/id_ed25519.pub
    print('\nAlternatively, download the file to your computer directly:')
    from google.colab import files
    files.download(gguf_file)
